# agentic-civic-resolution-app 
# Day 1 Deliverables

## Run Order

| # | Notebook | What it does |
|---|----------|-------------|
| 1 | "01_setup_and_ingest.py" | Creates Unity Catalog, schemas, fetches NYC 311 → **Bronze** Delta table |
| 2 | `02_bronze_to_silver.py` | Cleans, types, deduplicates Bronze → **Silver** Delta table |
| 3 | `03_synthetic_generator.py` | Generates synthetic complaints for dev/testing |

---

## Delta Table Schema

### `civicops.bronze.nyc311_raw`
Raw drop from NYC Open Data API — all original columns preserved.
Extra columns: `_ingested_at`, `_source`, `_ingestion_date`.
Partitioned by: `_ingestion_date`.

### `civicops.silver.complaints`

| Column | Type | Notes |
|--------|------|-------|
| `complaint_id` | STRING | Unique key from NYC 311 |
| `created_date` | TIMESTAMP | Complaint open date |
| `closed_date` | TIMESTAMP | Nullable |
| `due_date` | TIMESTAMP | Agency SLA deadline |
| `complaint_type` | STRING | Uppercased, trimmed |
| `descriptor` | STRING | Sub-category detail |
| `status` | STRING | OPEN / CLOSED / PENDING / IN PROGRESS |
| `resolution_description` | STRING | Nullable |
| `agency` | STRING | e.g. HPD, NYPD, DOT |
| `agency_name` | STRING | Full agency name |
| `borough_clean` | STRING | NULL instead of UNSPECIFIED |
| `city` | STRING | |
| `zip_code` | STRING | |
| `street_name` | STRING | |
| `latitude` | DOUBLE | Nullable |
| `longitude` | DOUBLE | Nullable |
| `has_geo` | BOOLEAN | Both lat/lon present |
| `is_closed` | BOOLEAN | Derived from status |
| `days_open` | INTEGER | Days to close (or since opened) |
| `open_year` | INTEGER | Partition key |
| `open_month` | INTEGER | |
| `open_dow` | INTEGER | 1=Sun … 7=Sat |
| `_silver_at` | TIMESTAMP | Pipeline run time |

Partitioned by: `open_year`, `borough_clean`.
Z-ordered by: `complaint_type`, `borough_clean`.

### `civicops.silver.complaints_synthetic`
Same schema as silver + `_is_synthetic BOOLEAN`.

---

## Day 1 Checklist

- [x] Unity Catalog + `civicops.bronze` + `civicops.silver` schemas
- [x] NYC 311 ingestion via Socrata API (paginated, last 2 years)
- [x] Bronze Delta table (raw, schema-on-read, partitioned)
- [x] Silver Delta table (typed, deduped, enriched, Z-ordered)
- [x] Synthetic complaint generator (50k default, configurable to 500k+)
- [x] Column coverage / profile reports

---

## Prerequisites

```bash
# Install in your Databricks cluster init script or notebook:
%pip install requests pandas numpy
```

- Databricks Runtime 13.3 LTS or higher (Unity Catalog support)
- Cluster with internet egress to `data.cityofnewyork.us`
- Unity Catalog metastore attached to workspace

---

## Tuning Tips

| Need | Change |
|------|--------|
| Faster initial load | Set `MAX_RECORDS = 100_000` in Notebook 1 |
| Full historical load | Set `MAX_RECORDS = None` (expect 30–40 min) |
| More synthetic data | Set `NUM_RECORDS = 500_000` in Notebook 3 |
| Incremental loads (Day 2+) | Switch `.mode("overwrite")` → `.mode("append")` + dedup merge |

---

## Next: Day 2

- Build **Gold** aggregation tables (complaint counts by type/borough/week)
- Wire **Genie** natural language Q&A space to Silver + Gold tables
- Add **AI/BI Dashboard** stubs (open complaints map, resolution time trends)
